# Qwen3-VL-8B ChartQA — QLoRA 訓練（Colab A100）

**用法**（本專案暫不走 GitHub，直接上傳 notebook）：
1. Colab → 檔案 → 上傳筆記本 → 選這個檔案
2. 執行階段 → 變更執行階段類型 → **A100 GPU**（T4 也能跑 smoke test，會自動用小 batch preset）
3. 左側 🔑 Secrets：確認已有 `HF_TOKEN`（write token）且「筆記本存取權」已開啟
4. 依序全部執行。**先跑 `SMOKE_TEST = True`（500 筆）確認 loss 有下降**，再改 `False` 全量訓練
5. 跑完記得：執行階段 → 中斷連線並刪除執行階段

**產物**（自動 push 到 HF Hub，本機不用手動搬檔案）：
- LoRA adapter → `<你的帳號>/qwen3vl-8b-chartqa-lora`（訓練中 checkpoint 也會定期 push，斷線重跑會自動 resume）
- 訓練曲線 `loss_curve.png` + `log_history.json` → 同一個 repo


In [ ]:
# 1. GPU 檢查
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
IS_A100 = "A100" in gpu
print(f"GPU: {gpu}  ->  preset: {'A100' if IS_A100 else 'T4/small'}")

In [ ]:
%%capture
# 2. 安裝相依（unsloth 官方 Colab 安裝區塊 + 版本 pin）
import os, re
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  # 先關閉：加速下載器在部分網路環境會靜默卡死；改回 "1" 前先確認這次全程沒卡住
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Colab/GCP 對 hf_xet 傳輸有已知卡死問題（huggingface_hub #3266, #4085）
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2
!pip uninstall -y -q hf_xet  # 雙保險：直接移除套件，避免 huggingface_hub 選用 Xet 傳輸而卡死

In [ ]:
# 3. 從 Colab Secrets 讀 HF_TOKEN 登入
from google.colab import userdata
from huggingface_hub import login, whoami
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
HF_USER = whoami()["name"]
print("logged in as:", HF_USER)

def _retry_hf(fn, *args, max_retries=5, base_delay=10.0, **kwargs):
    # HF Hub 的 Xet CDN 簽章在 Colab 上會間歇性失效，表現方式很多種（403、被包裝成
    # 「檔案不存在」的 OSError、離線載入殘缺快取的 AttributeError）。這裡單純重試；
    # 不能塞 force_download —— unsloth 內部是「先預下載、再離線載入」，離線階段
    # 收到 force_download 會直接 ValueError
    import time

    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [HF retry] attempt {attempt}/{max_retries} failed ({type(e).__name__}: {e}); retrying in {wait:.0f}s")
            time.sleep(wait)

def _prefetch_repo(repo_id, repo_type="model", max_retries=6, base_delay=10.0):
    # 真正的關鍵：先用整檔下載把 repo 抓齊進本機快取（走已證實可靠的路徑，失敗自動重試）。
    # unsloth/transformers 的「預下載失敗 -> 退回離線載入」流程在快取不完整時會炸出
    # 各種怪錯（checkpoint_files None / 檔案不存在）；快取抓齊之後離線載入必定成功
    import time
    from huggingface_hub import snapshot_download

    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id, repo_type=repo_type)
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [prefetch {repo_id}] attempt {attempt}/{max_retries} failed ({type(e).__name__}); retrying in {wait:.0f}s")
            time.sleep(wait)

def _prefetch_model_and_base(repo_id):
    # LoRA adapter repo 會連同 adapter_config.json 指到的底模一起預抓
    import json, os
    path = _prefetch_repo(repo_id)
    cfg = os.path.join(path, "adapter_config.json")
    if os.path.exists(cfg):
        base = json.load(open(cfg)).get("base_model_name_or_path")
        if base:
            print(f"[prefetch] adapter 底模: {base}")
            _prefetch_repo(base)
    return path

In [ ]:
# 4. 設定
SMOKE_TEST = True          # <<< 先 True 跑 500 筆看 loss；確認下降後改 False 全量
SEED = 3407

N_TRAIN   = 500 if SMOKE_TEST else 15_000   # 全量：train split 抽 1.5 萬筆
EPOCHS    = 1                                # loss 未收斂可改 2（resume 會接著跑）
MAX_LEN   = 2048

BASE_MODEL   = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
ADAPTER_REPO = f"{HF_USER}/qwen3vl-8b-chartqa-lora" + ("-smoke" if SMOKE_TEST else "")

# LoRA（unsloth 官方 Qwen3-VL notebook 預設）
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 16, 0

# batch presets：A100 40GB vs T4 16GB（等效 batch 都是 16）
if IS_A100:
    PER_DEVICE_BS, GRAD_ACCUM = 4, 4
else:
    PER_DEVICE_BS, GRAD_ACCUM = 1, 16

SAVE_STEPS = 25 if SMOKE_TEST else 100      # checkpoint push 頻率（斷線可 resume）
print(f"{ADAPTER_REPO=}  {N_TRAIN=}  {EPOCHS=}  bs={PER_DEVICE_BS}x{GRAD_ACCUM}")

## 資料前處理

> 與本機 repo 的 `src/chartqa_data.py` **保持同步**（notebook 是手動上傳、不 clone repo，所以這裡放一份相同邏輯；改動時兩邊都要改）。


In [ ]:
# 5. ChartQA -> Qwen3-VL chat messages（與 src/chartqa_data.py 同步）
from datasets import Dataset

DATASET_ID = "HuggingFaceM4/ChartQA"
ANSWER_INSTRUCTION = "Answer the question using a single word or phrase."
HUMAN, MACHINE = 0, 1

def _download_with_retry(filename, max_retries=6, base_delay=5.0):
    # HF 的 Xet CDN 簽章在 Colab 上會間歇性失效（同一檔案這次成功、下次 403），
    # 不是檔案本身壞掉；重新請求會拿到新的簽章 URL，重試就會過
    import time
    from huggingface_hub import hf_hub_download
    from huggingface_hub.errors import HfHubHTTPError

    for attempt in range(1, max_retries + 1):
        try:
            return hf_hub_download(DATASET_ID, filename, repo_type="dataset")
        except HfHubHTTPError as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [{filename}] attempt {attempt}/{max_retries} failed ({type(e).__name__}); retrying in {wait:.0f}s")
            time.sleep(wait)

def load_chartqa(split, n=None, human_or_machine=None, seed=42):
    # 直接抓該 split 的 parquet 檔（整檔循序下載＋失敗重試），不用 datasets.load_dataset()：
    # 它的 eager 模式會連其他 split 一起準備，streaming 模式則用 byte-range 讀取，
    # 都撞過 HF Xet CDN 在 Colab 上簽章失效的 403（結果證實是間歇性的，不是特定檔案壞掉）
    from huggingface_hub import HfApi

    files = sorted(f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset")
                   if f.startswith(f"data/{split}-"))
    local_paths = [_download_with_retry(f) for f in files]
    ds = Dataset.from_parquet(local_paths)
    if human_or_machine is not None:
        ds = ds.filter(lambda ex: ex["human_or_machine"] == human_or_machine)
    if n is not None and n < len(ds):
        ds = ds.shuffle(seed=seed).select(range(n))
    return ds

def get_answer(example):
    label = example["label"]
    return str(label[0]) if isinstance(label, list) else str(label)

def to_messages(example, include_answer=True):
    user_content = [
        {"type": "image", "image": example["image"]},
        {"type": "text", "text": f"{example['query']}\n{ANSWER_INSTRUCTION}"},
    ]
    messages = [{"role": "user", "content": user_content}]
    if include_answer:
        messages.append({"role": "assistant",
                         "content": [{"type": "text", "text": get_answer(example)}]})
    return messages

def convert_to_conversation(example):
    return {"messages": to_messages(example, include_answer=True)}

In [ ]:
# 6. 載入並轉換訓練資料（human + machine 都用，比照官方 train split 組成）
raw_train = load_chartqa("train", n=N_TRAIN, seed=SEED)
converted_dataset = [convert_to_conversation(ex) for ex in raw_train]
print(f"train rows: {len(converted_dataset)}")
ex0 = converted_dataset[0]["messages"]
print("sample user text:", ex0[0]["content"][1]["text"][:120])
print("sample answer   :", ex0[1]["content"][0]["text"])

In [ ]:
# 7. 載入模型 + LoRA
from unsloth import FastVisionModel

_prefetch_model_and_base(BASE_MODEL)   # 先抓齊檔案，unsloth 離線載入才不會踩到殘缺快取
model, tokenizer = _retry_hf(
    FastVisionModel.from_pretrained,
    BASE_MODEL,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,   # 圖表理解：視覺層一起微調
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)
model.print_trainable_parameters()

In [ ]:
# 8. Trainer（checkpoint 定期 push 到 HF Hub，斷線可 resume）
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = PER_DEVICE_BS,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 5,
        num_train_epochs = EPOCHS,
        learning_rate = 2e-4,
        logging_steps = 1 if SMOKE_TEST else 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = SEED,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = MAX_LEN,
        # --- checkpoint -> HF Hub（斷線 resume 用）---
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 1,
        push_to_hub = True,
        hub_model_id = ADAPTER_REPO,
        hub_strategy = "checkpoint",
        hub_private_repo = False,
    ),
)

In [ ]:
# 9. 訓練（若 Hub 上已有 last-checkpoint 就自動 resume）
import os
from huggingface_hub import HfApi, snapshot_download

resume_ckpt = None
api = HfApi()
try:
    if api.repo_exists(ADAPTER_REPO) and any(
        f.startswith("last-checkpoint/") for f in api.list_repo_files(ADAPTER_REPO)
    ):
        local = snapshot_download(ADAPTER_REPO, allow_patterns=["last-checkpoint/*"])
        resume_ckpt = os.path.join(local, "last-checkpoint")
        print("resuming from hub checkpoint:", resume_ckpt)
except Exception as e:
    print("no resumable checkpoint:", e)

import torch
torch.cuda.reset_peak_memory_stats()
trainer_stats = trainer.train(resume_from_checkpoint=resume_ckpt)

print(f"\ntrain time: {trainer_stats.metrics['train_runtime']/60:.1f} min")
print(f"peak VRAM : {torch.cuda.max_memory_reserved()/1024**3:.1f} GB")

In [ ]:
# 10. 訓練曲線 -> loss_curve.png + log_history.json，push 到同一個 repo
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

logs = [x for x in trainer.state.log_history if "loss" in x]
steps = [x["step"] for x in logs]
losses = [x["loss"] for x in logs]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(steps, losses, lw=1.5)
ax.set_xlabel("step"); ax.set_ylabel("train loss")
ax.set_title(f"Qwen3-VL-8B ChartQA QLoRA ({'smoke 500' if SMOKE_TEST else f'{N_TRAIN} samples'})")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("loss_curve.png", dpi=150)
with open("log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)

api.upload_file(path_or_fileobj="loss_curve.png", path_in_repo="loss_curve.png", repo_id=ADAPTER_REPO)
api.upload_file(path_or_fileobj="log_history.json", path_in_repo="log_history.json", repo_id=ADAPTER_REPO)
print(f"first loss={losses[0]:.4f}  last loss={losses[-1]:.4f}")
plt.show()

In [ ]:
# 11. 訓練後快速 sanity check（val 抽 3 筆）
FastVisionModel.for_inference(model)
val = load_chartqa("val", n=3, seed=SEED)
for ex in val:
    messages = to_messages(ex, include_answer=False)
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(ex["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    ans = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {ex['query']}\n  gold: {get_answer(ex)}\n  pred: {ans.strip()}\n")

In [ ]:
# 12. 最終 push：LoRA adapter + processor
model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
print(f"done -> https://huggingface.co/{ADAPTER_REPO}")

## 下一步

- **Smoke test（500 筆）**：確認上面 loss 曲線有明顯下降（例如從 ~2 降到 <0.5 量級）→ 把 `SMOKE_TEST = False` 改掉、執行階段重新啟動、再全部執行一次做全量訓練
- **全量訓練完成後**：確認 adapter 已推送到 HF Hub，接著執行 Phase 3 評估 notebook
- 跑完記得：執行階段 → **中斷連線並刪除執行階段**
